# Putting a number on it: ONI vs. your sea level record

Yesterday you built monthly sea level for your station and drew the big El Niño
winters over the seasonal cycle. Your eyes said the warm events ride high. Today
we test that against NOAA's standard yardstick for El Niño strength: the
**Oceanic Niño Index (ONI)**, the 3-month running mean of sea surface temperature
anomaly in the Niño 3.4 box (5°N–5°S, 120°–170°W). Five consecutive seasons at or
above +0.5 °C is the official definition of an El Niño event; at or below
−0.5 °C, La Niña.

The plan, with x = ONI and y = your station's sea level:

1. form monthly averages of y
2. remove the seasonal cycle from y (subtract the average of all Januarys from
   each January, and so on)
3. remove the trend from y
4. overlay time series of x and y
5. scatter plot x vs. y
6. compute the correlation

One given cell sits between steps 1 and 2: as in notebook 11, it splices NOAA
CO-OPS monthly means onto the end of the UHSLC record, so every station reaches
the present and the 2026–27 event counts in your correlation.

Unlike yesterday, this notebook does not run end-to-end as delivered: steps 1–6
are yours to write. Each is a few lines, each cell carries hints, and each step
ends with a checkpoint so you know whether to move on. A cell you haven't
finished stops at a `NotImplementedError` reminder — that's the notebook working
as intended, not a bug. The finished version is in
`reference/14_el_nino_oni_correlation_complete.ipynb` — wrestle here first.

In [ ]:
import json
from pathlib import Path
from urllib.request import urlopen, urlretrieve
from urllib.error import HTTPError, URLError

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

# Everything in the course runs relative to the project root, never to a
# personal path, so the same notebook works on every machine.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name in ("notebooks", "reference"):
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "README.md").exists(), "Open the course project folder first."

UHSLC_RAW = PROJECT_ROOT / "data" / "raw" / "uhslc"
UHSLC_RAW.mkdir(parents=True, exist_ok=True)

## 1. The x variable: ONI

The Climate Prediction Center publishes ONI as a plain text table, one row per
overlapping 3-month season since 1950:

```
 SEAS  YR   TOTAL   ANOM
  DJF 1950  25.01  -1.32
  JFM 1950  25.36  -1.20
```

`ANOM` is the index itself. Each season is stamped here at its **center month**
(DJF is a January value, JFM February, and so on), which turns the table into an
ordinary monthly time series we can line up against sea level. This loader is
given — the construction work starts with your y variable below.

In [ ]:
ONI_URL = "https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt"

# each 3-month season, mapped to the calendar month at its center
SEASON_CENTER = {"DJF": 1, "JFM": 2, "FMA": 3, "MAM": 4, "AMJ": 5, "MJJ": 6,
                 "JJA": 7, "JAS": 8, "ASO": 9, "SON": 10, "OND": 11, "NDJ": 12}


def load_oni():
    """ONI as a monthly pandas Series (deg C anomaly), stamped at month start."""
    table = pd.read_csv(ONI_URL, sep=r"\s+")
    center = table["SEAS"].map(SEASON_CENTER)
    idx = pd.to_datetime(dict(year=table["YR"], month=center, day=1))
    return pd.Series(table["ANOM"].to_numpy(), index=idx, name="ONI")


oni = load_oni()
print(f"ONI: {oni.index[0]:%Y-%m} to {oni.index[-1]:%Y-%m}, "
      f"latest value {oni.iloc[-1]:+.2f} deg C")

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(oni.index, oni, lw=0.8, color="k")
# the +/-0.5 degC thresholds that define El Nino / La Nina conditions
ax.axhline(0.5, color="crimson", ls="--", lw=0.8)
ax.axhline(-0.5, color="steelblue", ls="--", lw=0.8)
ax.fill_between(oni.index, 0.5, oni.clip(lower=0.5), color="crimson", alpha=0.5)
ax.fill_between(oni.index, -0.5, oni.clip(upper=-0.5), color="steelblue", alpha=0.5)
ax.set_ylabel("ONI (°C)")
ax.set_title("Oceanic Niño Index — red above +0.5 is El Niño territory")
plt.show()

## 2. The y variable: your station's daily sea level

Same data and loader as yesterday. If `data/raw/uhslc/` already has your file,
nothing downloads. Set `MY_SITE` to your group's station.

In [ ]:
STATIONS = {
    # name: UHSLC daily file        record ends
    "San Diego":     "d569.nc",   # 2026 —— the in-class demo
    "La Jolla":      "d554.nc",   # 2026
    "Los Angeles":   "d567a.nc",  # 2021
    "Port San Luis": "d565a.nc",  # 2021
    "Monterey":      "d555a.nc",  # 2021
    "San Francisco": "d551.nc",   # 2026
    "Humboldt Bay":  "d576a.nc",  # 2021
    "Crescent City": "d556.nc",   # 2026
    "South Beach":   "d592.nc",   # 2026 —— Oregon: sea level only
    "Astoria":       "d572a.nc",  # 2024 —— Oregon: sea level only
}

MY_SITE = "San Diego"  # TODO: change to your group's station

UHSLC_BASE = "https://uhslc.soest.hawaii.edu/data/netcdf"
ARCHIVES = ["fast/daily", "rqds/pacific/daily"]


def fetch_station(filename):
    """Download one UHSLC daily NetCDF file into data/raw/uhslc, if not there."""
    target = UHSLC_RAW / filename
    if target.exists():
        return target  # already on disk: never re-download raw data
    for archive in ARCHIVES:
        try:
            urlretrieve(f"{UHSLC_BASE}/{archive}/{filename}", target)
            print(f"downloaded {filename} from {archive}")
            return target
        except HTTPError:
            continue
    raise FileNotFoundError(f"{filename} not found in any UHSLC archive")


def load_daily(path):
    """Daily sea level as a pandas Series in cm, plus station name and latitude."""
    with xr.open_dataset(path) as ds:
        name = ds["station_name"].values.astype(str).item()
        lat = float(ds["lat"].values.squeeze())
        daily = ds["sea_level"].squeeze("record_id", drop=True).to_series() * 0.1
    return daily, name, lat


daily, name, lat = load_daily(fetch_station(STATIONS[MY_SITE]))
print(f"{name} ({lat:.2f}°N): {daily.index[0].date()} to {daily.index[-1].date()}")

## 3. Step 1 — monthly averages

ONI is monthly, so y must be monthly too. Turn the daily series into
calendar-month means, and blank any month with fewer than 15 valid days — an
average built from a handful of days is not a trustworthy monthly mean.

**Hints:** `daily.resample("MS")` groups a daily series by calendar month
(`"MS"` stamps each group at month start, matching how we stamped ONI).
`.mean()` on the grouped object averages each month; `.count()` on the same
grouped object tells you how many real observations each month has.

In [ ]:
grouped = daily.resample("MS")
monthly = ...          # TODO: mean of each month, from `grouped`
...                    # TODO: set months with fewer than 15 valid days to np.nan

if monthly is ...:
    raise NotImplementedError("finish the TODO lines above, then re-run this cell")
# checkpoint: one value per calendar month, spanning the whole record
print(f"{len(monthly)} months, {monthly.index[0]:%Y-%m} to {monthly.index[-1]:%Y-%m}, "
      f"{monthly.isna().sum()} blanked")

## 4. Extend to the present with NOAA CO-OPS (given)

Half the transect's UHSLC files end in 2021 or 2024, but ONI runs to last month
and the 2026–27 event is in progress — months your correlation should not have
to do without. NOAA still operates every one of these gauges, so this cell (the
same splice as notebook 11) appends CO-OPS verified monthly means, plus a
provisional average of the current month, to the end of your `monthly` series.
The two sources sit on different datums; the median difference over their shared
months removes it, and the printed scatter is the sanity check — a centimeter or
so means the station pairing is right.

In [ ]:
if monthly is ...:
    raise NotImplementedError("finish Step 1 first, then re-run this cell")

COOPS_IDS = {   # NOAA CO-OPS ids for the same gauges
    "San Diego": "9410170", "La Jolla": "9410230", "Los Angeles": "9410660",
    "Port San Luis": "9412110", "Monterey": "9413450", "San Francisco": "9414290",
    "Humboldt Bay": "9418767", "Crescent City": "9419750",
    "South Beach": "9435380", "Astoria": "9439040",
}
COOPS_API = "https://api.tidesandcurrents.noaa.gov/api/prod/datagetter"
TODAY = pd.Timestamp.today()


def coops_request(station_id, product, begin, end):
    """One call to the NOAA CO-OPS data API, returning the rows as a list."""
    url = (f"{COOPS_API}?product={product}&begin_date={begin}&end_date={end}"
           f"&datum=STND&station={station_id}&time_zone=GMT&units=metric&format=json")
    with urlopen(url, timeout=60) as response:
        return json.load(response).get("data", [])


def extend_with_coops(monthly, station_id, label=""):
    """Append offset-adjusted CO-OPS months after the UHSLC record ends."""
    # 1) verified monthly means since 2017 (meters on NOAA's datum -> cm)
    rows = coops_request(station_id, "monthly_mean", "20170101",
                         TODAY.strftime("%Y%m%d"))
    idx = pd.to_datetime([f"{r['year']}-{int(r['month']):02d}-01" for r in rows])
    coops = pd.Series([float(r["MSL"]) * 100 for r in rows], index=idx)

    # 2) the current month so far, from the preliminary 6-minute feed
    recent = coops_request(station_id, "water_level",
                           TODAY.strftime("%Y%m01"), TODAY.strftime("%Y%m%d"))
    values = [float(r["v"]) for r in recent if r.get("v")]
    if len(values) >= 7 * 240:  # only with at least a week of data
        coops[pd.Timestamp(TODAY.strftime("%Y-%m-01"))] = np.mean(values) * 100

    # 3) align the two sources: the datum difference is a constant, so the
    #    median difference over shared months estimates it robustly
    both = pd.concat([monthly, coops], axis=1, keys=["uhslc", "coops"]).dropna()
    offset = (both["uhslc"] - both["coops"]).median()
    print(f"{label:<16} offset {offset:+6.1f} cm over {len(both)} shared months, "
          f"scatter {(both['uhslc'] - both['coops'] - offset).std():4.1f} cm, "
          f"through {coops.index[-1]:%Y-%m}")

    # 4) keep UHSLC as-is and append only the months it doesn't have
    tail = coops[coops.index > monthly.last_valid_index()] + offset
    return pd.concat([monthly[: monthly.last_valid_index()], tail])


monthly = extend_with_coops(monthly, COOPS_IDS[MY_SITE], MY_SITE)

## 5. Step 2 — remove the seasonal cycle

Sea level here breathes with the seasons (steric heating in summer, storms and
pressure in winter), and that cycle is as large as the El Niño signal we are
hunting. Remove it: subtract the average of all Januarys from every January, all
Februarys from every February, and so on. What survives is the **anomaly** — the
part of each month that the calendar cannot explain.

**Hints:** `monthly.index.month` gives each row's calendar month number (1–12).
`monthly.groupby(monthly.index.month)` groups by it, and `.transform("mean")`
returns those 12 group means stretched back to the full length of the series —
exactly the thing to subtract. (Yesterday's `seasonal_climatology` used the same
groupby with `.mean()`; `transform` is the same idea, shaped for subtraction.)

In [ ]:
climatology = ...      # TODO: each month's calendar-month mean, full length
anomaly = ...          # TODO: subtract it

if anomaly is ...:
    raise NotImplementedError("finish the TODO lines above, then re-run this cell")
# checkpoint: the anomaly's own seasonal cycle should be zero by construction
residual_cycle = anomaly.groupby(anomaly.index.month).mean()
print(f"largest leftover monthly mean: {residual_cycle.abs().max():.2e} cm (want ~0)")

## 6. Step 3 — remove the trend

A century of relative sea level rise is still in the anomaly, and over 75 shared
years it would masquerade as correlation with anything else that drifts. Fit a
straight line against time and subtract it, exactly as yesterday.

**Hints:** build a time axis in fractional years from the index
(`anomaly.index.year + (anomaly.index.month - 0.5) / 12`). `np.polyfit(x, y, 1)`
returns the pair (slope, intercept), but cannot digest NaNs — mask them out of
the fit with `anomaly.notna()`, then subtract the fitted line from the *whole*
series. The slope is in cm/yr; multiply by 10 for mm/yr to compare with
yesterday's number.

In [ ]:
if anomaly is ...:
    raise NotImplementedError("finish Step 2 first, then re-run this cell")
years = anomaly.index.year + (anomaly.index.month - 0.5) / 12
valid = anomaly.notna()
fit = ...                # TODO: np.polyfit on the valid months only -> (slope, intercept)
sl = ...                 # TODO: anomaly minus the fitted line, fit[0] * years + fit[1]

if sl is ...:
    raise NotImplementedError("finish the TODO lines above, then re-run this cell")
# checkpoint: the trend should match what you found (and NOAA publishes) yesterday
print(f"{name} trend: {fit[0] * 10:.1f} mm/yr; detrended series centered at "
      f"{sl.mean():+.2f} cm")

## 7. Step 4 — overlay the two time series

Now look before you compute. ONI is in °C and sea level in cm, so give each its
own y-axis and let the shapes do the talking.

**Hints:** `ax2 = ax.twinx()` makes a second y-axis sharing the same time axis.
Plot ONI on one, `sl` on the other, in different colors, and restrict the view to
1950 onward (`sl["1950":]`) since that is where ONI begins. Where are 1982–83,
1997–98, 2015–16, 2023–24 in each curve?

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
...                    # TODO: ONI on ax, in its own color

ax2 = ax.twinx()       # second y-axis, same time axis
...                    # TODO: sl (from 1950 on) on ax2

ax.set_ylabel("ONI (°C)")
ax2.set_ylabel("Sea level anomaly (cm)")
ax.set_title(f"ONI and {name} sea level anomaly")
plt.show()

## 8. Step 5 — the scatter plot

The overlay shows *when* the two move together; the scatter shows *how much*, one
point per shared month.

**Hints:** the two series have different spans and gaps, so align them first:
`pd.concat([oni, sl.rename("sl")], axis=1).dropna()` keeps exactly the months
where both exist. With ~900 points, `ax.scatter(..., s=8, alpha=0.4)` keeps the
cloud readable.

In [ ]:
both = ...             # TODO: align the two series, drop months missing either

if both is ...:
    raise NotImplementedError("finish the TODO line above, then re-run this cell")
print(f"{len(both)} shared months, {both.index[0]:%Y-%m} to {both.index[-1]:%Y-%m}")

fig, ax = plt.subplots(figsize=(5.5, 5))
...                    # TODO: scatter ONI (x) against sea level anomaly (y)
ax.set_xlabel("ONI (°C)")
ax.set_ylabel("Sea level anomaly (cm)")
ax.set_title(name)
plt.show()

## 9. Step 6 — the correlation

One number for the board.

**Hints:** `both["ONI"].corr(both["sl"])` gives Pearson's r (it ignores nothing —
you already dropped the gaps). While you are at it, `np.polyfit` on the two
columns gives the regression slope: cm of sea level per °C of ONI, a number with
physical units your neighbors can compare.

In [ ]:
r = ...                     # TODO: Pearson correlation of the two columns
slope_cm_per_degC = ...     # TODO: regression slope, cm per degC of ONI

if r is ...:
    raise NotImplementedError("finish the TODO lines above, then re-run this cell")
print(f"{name}: r = {r:.2f}  (r² = {r**2:.2f}), "
      f"{slope_cm_per_degC:.1f} cm per °C of ONI")

## 10. Interpret

**For the class board:** station, latitude, r, r², and the regression slope
(cm/°C). Then, as a group:

1. Is the correlation positive or negative, and what physics carries a warm
   tropical Pacific to sea level at *your* latitude? (Think back to this
   morning: coastally trapped waves, thermosteric warming, and the shifted storm
   track are all candidates.)
2. r² is the fraction of monthly variance ONI explains. What is doing the rest?
3. Look at the scatter's outliers — months far off the trend. Pick one, find its
   date in `both`, and check what was happening (a storm? a gap?).
4. Prediction before the board fills in: will r grow, shrink, or hold as the
   stations march north from San Diego to Astoria?

## 11. If you finish early

- **Match the smoothing.** ONI is a 3-month running mean but your y is raw
  monthly. Smooth it the same way
  (`sl.rolling(3, center=True, min_periods=2).mean()`) and recompute r. For San
  Diego this lifts r from 0.65 to 0.70 — why should matching the smoothing help?
- **Lead or lag?** `oni.shift(k)` moves ONI k months later. Loop k from −6 to +6
  and find where r peaks. Does the ocean here respond to the tropics instantly,
  or late?
- **Waves too?** Notebook 12 gave you a monthly wave record. Does ONI correlate
  with wave energy at your station the way it does with sea level?

## Data notes

ONI from the NOAA Climate Prediction Center
([oni.ascii.txt](https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt), ERSSTv5
Niño 3.4 anomalies, centered 30-year base periods). UHSLC daily tide gauge data
(Caldwell, Merrifield, and Thompson 2015, doi:10.7289/V5V40S7W), extended to the
present with NOAA CO-OPS verified monthly means plus a provisional current-month
estimate; raw files stay in `data/raw/` and out of Git. The finished analysis:
`reference/14_el_nino_oni_correlation_complete.ipynb`.